In [1]:
# Colab mount

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Sanity checks for features_* parquet files
This notebook section verifies that the four segment feature files:
- include the required **context months** (CHN: 2025-08; USA: 2025-07),
- preserve **unique keys** (origin, destination, hs6, hs4, trade_flow, month),
- have **fresh** lags (e.g., `lag_1(t) == y(t-1)`),
- and that the **context month** has usable feature inputs (lags/MAs not missing).


In [2]:
import os
import pandas as pd
import numpy as np

# === Paths ===
BASE_DIR = "/content/drive/MyDrive/ai4trade"  # adjust if needed
DATA_FEATURES_DIR = f"{BASE_DIR}/data/features"

FILES = {
    "features_CHN_export.parquet": {"origin": "CHN", "flow": "Export", "t_last_context": "2025-08-01"},
    "features_CHN_import.parquet": {"origin": "CHN", "flow": "Import", "t_last_context": "2025-08-01"},
    "features_USA_export.parquet": {"origin": "USA", "flow": "Export", "t_last_context": "2025-07-01"},
    "features_USA_import.parquet": {"origin": "USA", "flow": "Import", "t_last_context": "2025-07-01"},
}

KEYS = ["origin","destination","hs6","hs4","trade_flow","month"]

def _norm_month(series):
    s = pd.to_datetime(series, errors="coerce")
    return s.dt.to_period("M").dt.to_timestamp()

def _dedupe_cols(df):
    if df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated(keep="first")]
    return df

def coverage_checks(df, fname, t_last_context):
    out = {}
    df["month"] = _norm_month(df["month"])
    out["rows"] = len(df)
    out["month_min"] = str(df["month"].min()) if len(df) else None
    out["month_max"] = str(df["month"].max()) if len(df) else None
    out["n_months"]  = int(df["month"].nunique())
    out["has_context_month"] = pd.Timestamp(t_last_context) in set(df["month"].unique())
    out["n_destinations"] = int(df["destination"].nunique())
    out["n_hs6"] = int(df["hs6"].nunique())
    out["flows"] = sorted(df["trade_flow"].astype(str).unique().tolist())
    print(f"\n[{fname}] Coverage")
    for k,v in out.items():
        print(f"  {k}: {v}")
    return df

def uniqueness_check(df, fname):
    dup_count = df.duplicated(KEYS, keep=False).sum()
    if dup_count > 0:
        raise AssertionError(f"[{fname}] Found {dup_count} duplicate key rows on {KEYS}")
    print(f"[{fname}] Key uniqueness: OK ({len(df)} unique rows)")

def lag_freshness_check(df, fname, sample_groups=5):
    # Quick check: for a few random series, does lag_1(t) == y(t-1)?
    if "lag_1" not in df.columns:
        print(f"[{fname}] lag_1 not present; skipping freshness check.")
        return
    ok = True
    keys = ["origin","destination","hs6","trade_flow"]
    # choose some largest groups to increase chance of a meaningful test
    sizes = df.groupby(keys).size().sort_values(ascending=False).head(50)
    groups = sizes.index.tolist()[:sample_groups]
    for g in groups:
        sub = df[(df[keys[0]]==g[0]) & (df[keys[1]]==g[1]) & (df[keys[2]]==g[2]) & (df[keys[3]]==g[3])].sort_values("month")
        if len(sub) < 3:
            continue
        # align (y shifted forward by 1) vs lag_1
        y_prev = sub["y"].shift(1)
        lag1   = sub["lag_1"]
        comp   = ( (y_prev.isna() & lag1.isna()) | (np.isclose(y_prev, lag1, equal_nan=True)) )
        if not comp.all():
            ok = False
            bad = sub.loc[~comp, ["month","y","lag_1"]].head(5)
            print(f"[{fname}] lag_1 mismatch in group {g} (showing first few rows):\n{bad}")
            break
    print(f"[{fname}] lag_1 freshness: {'OK' if ok else 'MISMATCH FOUND'}")

def context_feature_readiness(df, fname, t_last_context):
    # On the context month, we want lags/MAs mostly present (since they use <= t-1 data).
    ctx = df[df["month"] == pd.Timestamp(t_last_context)].copy()
    if ctx.empty:
        print(f"[{fname}] No context rows present; cannot assess feature readiness.")
        return
    feature_cols = [c for c in ["lag_1","lag_2","lag_3","lag_6","lag_12","ma_3","ma_6","ma_12"] if c in df.columns]
    if not feature_cols:
        print(f"[{fname}] No lag/MA columns present to assess.")
        return
    pct_nonnull = (1.0 - ctx[feature_cols].isna().mean()).sort_values(ascending=False)
    print(f"[{fname}] Context-month non-null ratio for key features:")
    print((pct_nonnull*100).round(1).astype(str) + "%")


In [3]:
for fname, cfg in FILES.items():
    path = os.path.join(DATA_FEATURES_DIR, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")

    # Read minimal first to lower memory if needed; switch to full if small enough in your environment
    df = pd.read_parquet(path)
    df = _dedupe_cols(df)

    # 1) Coverage + context-month presence
    df = coverage_checks(df, fname, cfg["t_last_context"])

    # 2) Key uniqueness
    uniqueness_check(df, fname)

    # 3) Lag freshness on a few groups
    lag_freshness_check(df, fname, sample_groups=5)

    # 4) Context-month feature readiness (are lags/MAs available at t?)
    context_feature_readiness(df, fname, cfg["t_last_context"])



[features_CHN_export.parquet] Coverage
  rows: 3059124
  month_min: 2023-01-01 00:00:00
  month_max: 2025-08-01 00:00:00
  n_months: 32
  has_context_month: True
  n_destinations: 30
  n_hs6: 5283
  flows: ['Export']
[features_CHN_export.parquet] Key uniqueness: OK (3059124 unique rows)
[features_CHN_export.parquet] lag_1 freshness: OK
[features_CHN_export.parquet] Context-month non-null ratio for key features:
lag_1     99.7%
ma_12     99.7%
ma_6      99.7%
ma_3      99.7%
lag_2     99.5%
lag_3     99.1%
lag_6     98.0%
lag_12    95.1%
dtype: object

[features_CHN_import.parquet] Coverage
  rows: 1261279
  month_min: 2023-01-01 00:00:00
  month_max: 2025-08-01 00:00:00
  n_months: 32
  has_context_month: True
  n_destinations: 29
  n_hs6: 5118
  flows: ['Import']
[features_CHN_import.parquet] Key uniqueness: OK (1261279 unique rows)
[features_CHN_import.parquet] lag_1 freshness: OK
[features_CHN_import.parquet] Context-month non-null ratio for key features:
lag_1     99.2%
ma_12     